# 01 — Inspect one case before modeling
Run one cell at a time and write a sentence under each output saying what it means.

**Checkpoint 1**: kernel is `.venv`; record `sys.executable` and the device check below.

In [ ]:
import sys, torch, monai, nibabel
print(sys.executable)
print(torch.__version__, monai.__version__, nibabel.__version__)
print('cuda:', torch.cuda.is_available())

## Dataset provenance (Checkpoint 2)
Open `data/raw/Task01_BrainTumour/dataset.json` and record: release, channel names, label names, license, number of training images. Do not infer channel order from filenames.

In [ ]:
import json
from pathlib import Path
root = Path('../data/raw/Task01_BrainTumour')
meta = json.loads((root / 'dataset.json').read_text(encoding='utf-8-sig'))
for k in ('name', 'release', 'licence', 'modality', 'labels', 'numTraining', 'numTest'):
    print(k, ':', meta.get(k))
imgs = sorted(p.name for p in (root/'imagesTr').glob('*.nii.gz') if not p.name.startswith('._'))
labs = sorted(p.name for p in (root/'labelsTr').glob('*.nii.gz') if not p.name.startswith('._'))
print(len(imgs), len(labs), 'paired:', set(imgs) == set(labs))

In [ ]:
import nibabel as nib
import numpy as np
import matplotlib.pyplot as plt

case = 'BRATS_001'  # replace with an ID you found
image = nib.load(root / 'imagesTr' / f'{case}.nii.gz')
label = nib.load(root / 'labelsTr' / f'{case}.nii.gz')
print('image:', image.shape, image.header.get_zooms())
print('label:', label.shape, label.header.get_zooms())
print('orientation:', nib.aff2axcodes(image.affine))
print('label values:', np.unique(label.get_fdata(dtype=np.float32)))

## Investigations to complete
1. One slice from each of the 4 channels with visible sequence labels.
2. Intensity min/max and finite count per channel (MRI values are not RGB).
3. Overlay the label on a slice at the same index; verify alignment (shape, affine).
4. Tumour voxel counts per label (0/1/2/3) for several cases.

In [ ]:
x = np.asarray(image.dataobj, dtype=np.float32)
y = np.asarray(label.dataobj).astype(int)
z = int(np.argmax((y > 0).sum(axis=(0, 1))))  # slice with most tumour
names = ['FLAIR', 'T1w', 'T1gd', 'T2w']  # verify against dataset.json
fig, ax = plt.subplots(1, 5, figsize=(18, 4))
for c in range(4):
    ax[c].imshow(x[..., z, c].T, cmap='gray', origin='lower'); ax[c].set_title(names[c]); print(names[c], x[..., c].min(), x[..., c].max(), np.isfinite(x[..., c]).sum())
ax[4].imshow(x[..., z, 0].T, cmap='gray', origin='lower'); ax[4].imshow(np.ma.masked_equal(y[..., z].T, 0), alpha=0.5, origin='lower'); ax[4].set_title('label overlay')
plt.show()